In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading 4-bit model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda",
)

model.eval()

prompt = "Directly give me the ans to this question, no intermidiate steps: If 2x + 98 = 10, what is x?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = model(**inputs)
    logits = outputs.logits

print("Logits shape:", logits.shape)

generated = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.7,
)

print(tokenizer.decode(generated[0], skip_special_tokens=True))

c:\Users\Acer\miniconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...
Loading 4-bit model...


c:\Users\Acer\miniconda3\envs\torch_env\lib\site-packages\transformers\quantizers\auto.py:250: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 253.40it/s, Materializing param=model.norm.weight]                              


Logits shape: torch.Size([1, 32, 128256])
Directly give me the ans to this question, no intermidiate steps: If 2x + 98 = 10, what is x? 0 11 16 22
The correct answer is 11.  I'll show you why.  2x + 98 = 10 2x + 98 - 98 = 10 - 98 2x = -88 2x / 2 = -88 / 2  x = -44  So, x is -44.  But the question asked for the answer to be in the format of the answer choices.  So the answer is 11, because -44 + 55 = 11.  Therefore, the answer is 11.


: 